In [38]:
import os
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

In [62]:
DB_DIR = "./chroma_db_cosine"
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"batch_size": 32, "normalize_embeddings": True},
)


In [6]:
# os.environ["GOOGLE_API_KEY"] = "your_API_KEY"


In [63]:
#llm
load_dotenv() #load GOOGLE_API_KEY from .env file
llm = ChatGoogleGenerativeAI( model = "gemini-3.5-flash-lite", temperature =0)
response = llm.invoke("Reply with only the words: Connection successful!")
print(response.content)

[{'type': 'text', 'text': 'Connection successful!', 'extras': {'signature': 'EowBCokBAWkUfRPbJ4PxltFgvG0fROct1xpLwzCg+fPekVXlLRR439twkCUHNNKtKSP2r3y580rSLh67i2Ko4jTdt1pl0BZiP16b4hWfitSbaPAm6owpdz3UCZuajPr1BIw/N0Yk6w7grTOhWw7EyeJsjbvdmksqG193M5Uastkb/KZYTeNwCC8CIz3G3uU='}}]


In [66]:
#load vectorstore and setup retriever
vectorstore = Chroma( persist_directory = DB_DIR, embedding_function= embeddings,collection_metadata={"hnsw:space": "cosine"} )
print("✅ Total chunks loaded:", vectorstore._collection.count())

# retriever = vectorstore.as_retriever(search_kwargs= {"k":5})
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "score_threshold": 0.4,  # Minimum similarity score (0.0 to 1.0)
        "k": 5                   # Optional: max number of documents to return
    }
)


✅ Total chunks loaded: 1455


In [67]:
prompt = ChatPromptTemplate.from_template(
    """ Answer the question based ONLY on the following context
    
    Rules:
    - Cite the relevant page number(s) inline as [Page X] when stating facts.
    - Only cite pages that actually contributed to your answer.
    - At the very end of your response, add a single line: 
      "📄 Sources: Referenced from page(s) X, Y."

    Context: {context}

    Question: {question}
    
    """)

# def format_docs(docs):
#     return "\n\n".join(doc.page_content for doc in docs)
def format_docs(docs):
    formatted = []
    for doc in docs:
        # Remove '+ 1' if your Chroma metadata already uses 1-indexed pages
        page = doc.metadata.get("page", None)
        page_label = f"Page {page + 1}" if page is not None else "Unknown Page"
        formatted.append(f"[{page_label}]\n{doc.page_content}")
    return "\n\n".join(formatted)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [68]:
question = "how we train a model?"
print("\nAnswer:\n", rag_chain.invoke(question))



Answer:
 Based on the provided context, model-based learning algorithms are trained by searching for optimal values for the model parameters so that the model generalizes well to new instances [Page 494]. Typically, this is done by minimizing a cost function that measures how poorly the system makes predictions on the training data, along with a penalty for model complexity if the model is regularized [Page 494]. Additionally, online learning systems generally require training models on a regular basis using fresh data, a process that should be automated as much as possible [Page 98].

📄 Sources: Referenced from page(s) 98, 494.
